# Kasus A - Deteksi Penipuan Rekening (Bank Account Fraud)

**Objective:** Membangun model klasifikasi biner yang menandai pembukaan rekening berpotensi fraud, dengan fokus menangkap sebanyak mungkin fraud pada batas false positive yang dapat ditoleransi bisnis.

**Data source:** `Base.csv` (Bank Account Fraud / BAF dataset, NeurIPS 2022), 1.000.000 baris dan 32 kolom, termasuk label `fraud_bool` dan kolom waktu `month` (0-7).

**Metric:** True Positive Rate (recall) pada False Positive Rate <= 5% - ditulis TPR@FPR5%. Accuracy tidak dipakai karena kelas fraud sangat tidak seimbang (~1,1%); biaya meloloskan fraud jauh lebih besar daripada biaya review manual tambahan. ROC AUC hanya dilaporkan sebagai diagnostik sekunder.

**Split rationale:** Split berbasis waktu, bukan acak. Train = `month` 0, 1, 2; validation = `month` 3. Pola fraud bergeser seiring waktu (perubahan taktik serangan dan perilaku pengguna), jadi evaluasi pada bulan setelah periode training lebih realistis untuk pemakaian ke depan. Split acak membocorkan pola temporal dan cenderung menaikkan skor secara artifisial. Bulan 4-7 tidak dipakai agar validation tetap satu periode yang belum pernah dilihat model.

**Catatan sensitif:** `customer_age` dan beberapa atribut demografis lain berpotensi sensitif. Proyek ini tidak membuat klaim fairness.


In [1]:
from pathlib import Path
import pickle
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("data/Base.csv")
MODEL_PATH = Path("model.pkl")

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di {DATA_PATH.resolve()}. "
        "Letakkan Base.csv di direktori yang sama dengan notebook ini."
    )

df = pd.read_csv(DATA_PATH)
df = df[df["month"].isin([0, 1, 2, 3])].reset_index(drop=True)

assert set(df["month"].unique()) == {0, 1, 2, 3}, sorted(df["month"].unique())
print(f"Rows after month 0-3 filter: {len(df):,}")


Rows after month 0-3 filter: 547,975


In [3]:
print("Shape:", df.shape)

fraud_count = int(df["fraud_bool"].sum())
fraud_rate = df["fraud_bool"].mean()
print(f"Fraud: {fraud_count:,} / {len(df):,} ({fraud_rate:.4%})")

print("\nMonth distribution:")
print(df["month"].value_counts().sort_index().to_string())

missing = df.isna().sum()
print("\nMissing counts (raw):")
print(missing[missing > 0].to_string() if (missing > 0).any() else "none")

cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print("\nCategorical cardinalities:")
for col in cat_cols:
    print(f"  {col}: {df[col].nunique()}")

age_summary = (
    df.groupby("fraud_bool")["customer_age"]
    .agg(count="size", mean="mean", std="std", min="min", median="median", max="max")
    .round(2)
)
print("\nAge by label:")
print(age_summary.to_string())


Shape: (547975, 32)
Fraud: 5,288 / 547,975 (0.9650%)

Month distribution:
month
0    132440
1    127620
2    136979
3    150936

Missing counts (raw):
none

Categorical cardinalities:
  payment_type: 5
  employment_status: 7
  housing_status: 7
  source: 2
  device_os: 5

Age by label:
             count   mean    std  min  median  max
fraud_bool                                        
0           542687  33.64  12.00   10    30.0   90
1             5288  41.26  13.25   10    40.0   90


In [4]:
# Datasheet: -1 = missing HANYA untuk enam kolom ini.
# Jangan ubah nilai negatif sah di kolom lain (mis. intended_balcon_amount, velocity_6h).
SENTINEL_COLS = [
    "prev_address_months_count",
    "current_address_months_count",
    "credit_risk_score",
    "bank_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w",
]

for col in SENTINEL_COLS:
    assert (df[col] == -1).any(), f"expected -1 sentinel in {col}"
    df[col] = df[col].mask(df[col] == -1, np.nan)

print("Converted -1 -> NaN, resulting missing counts:")
print(df[SENTINEL_COLS].isna().sum().to_string())


Converted -1 -> NaN, resulting missing counts:
prev_address_months_count       377557
current_address_months_count      2309
credit_risk_score                  406
bank_months_count               136894
session_length_in_minutes          995
device_distinct_emails_8w          226


In [5]:
train_df = df[df["month"] < 3].reset_index(drop=True)
valid_df = df[df["month"] == 3].reset_index(drop=True)

TARGET = "fraud_bool"
EXCLUDED = ["fraud_bool", "month", "device_fraud_count"]
FEATURES = [col for col in df.columns if col not in EXCLUDED]

y_train = train_df[TARGET]
y_valid = valid_df[TARGET]
X_train = train_df[FEATURES]
X_valid = valid_df[FEATURES]

assert set(train_df["month"]) == {0, 1, 2}
assert set(valid_df["month"]) == {3}
assert set(train_df["month"]).isdisjoint(set(valid_df["month"]))
assert not set(EXCLUDED) & set(FEATURES)
assert y_train.nunique() == 2 and y_valid.nunique() == 2
assert len(X_train) + len(X_valid) == len(df)
assert df["device_fraud_count"].nunique() == 1

print(f"Train (month<3): {X_train.shape} | fraud={int(y_train.sum()):,} ({y_train.mean():.4%})")
print(f"Valid (month==3): {X_valid.shape} | fraud={int(y_valid.sum()):,} ({y_valid.mean():.4%})")
print(f"Features used: {len(FEATURES)}")
print("Excluded from X:", ", ".join(EXCLUDED))


Train (month<3): (397039, 29) | fraud=3,896 (0.9813%)
Valid (month==3): (150936, 29) | fraud=1,392 (0.9222%)
Features used: 29
Excluded from X: fraud_bool, month, device_fraud_count


## Task 3 - Baseline & Model Utama

Dua model, tanpa tuning:
1. **Baseline**: Logistic Regression + `ColumnTransformer` (median impute + scale untuk numerik, most-frequent impute + one-hot untuk kategorikal).
2. **Model utama**: LightGBM dengan enam fitur kategorikal eksplisit; level kategori diselaraskan dari nilai yang teramati di train+valid, bukan dari label.

ROC AUC dilaporkan hanya sebagai diagnostik; metrik utama (TPR@FPR5%) dihitung di task berikutnya.

In [6]:
from sklearn.metrics import roc_auc_score

# email_is_free adalah biner integer, tetapi diperlakukan sebagai kategorikal
# karena tidak punya urutan/arti numerik kontinu.
CAT_COLS = df.select_dtypes(include=["object", "string"]).columns.tolist()
if "email_is_free" not in CAT_COLS:
    CAT_COLS.append("email_is_free")
NUM_COLS = [col for col in FEATURES if col not in CAT_COLS]

assert set(CAT_COLS) | set(NUM_COLS) == set(FEATURES)
assert not set(CAT_COLS) & set(NUM_COLS)

baseline = Pipeline(
    [
        (
            "prep",
            ColumnTransformer(
                [
                    (
                        "num",
                        Pipeline(
                            [
                                ("impute", SimpleImputer(strategy="median")),
                                ("scale", StandardScaler()),
                            ]
                        ),
                        NUM_COLS,
                    ),
                    (
                        "cat",
                        Pipeline(
                            [
                                ("impute", SimpleImputer(strategy="most_frequent")),
                                ("onehot", OneHotEncoder(handle_unknown="ignore")),
                            ]
                        ),
                        CAT_COLS,
                    ),
                ]
            ),
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
            ),
        ),
    ]
)

baseline.fit(X_train, y_train)
valid_proba_baseline = baseline.predict_proba(X_valid)[:, 1]
auc_baseline = roc_auc_score(y_valid, valid_proba_baseline)

print(f"Numeric ({len(NUM_COLS)}): {', '.join(NUM_COLS)}")
print(f"Categorical ({len(CAT_COLS)}): {', '.join(CAT_COLS)}")
print(f"Baseline ROC AUC (valid): {auc_baseline:.4f}")

Numeric (23): income, name_email_similarity, prev_address_months_count, current_address_months_count, customer_age, days_since_request, intended_balcon_amount, zip_count_4w, velocity_6h, velocity_24h, velocity_4w, bank_branch_count_8w, date_of_birth_distinct_emails_4w, credit_risk_score, phone_home_valid, phone_mobile_valid, bank_months_count, has_other_cards, proposed_credit_limit, foreign_request, session_length_in_minutes, keep_alive_session, device_distinct_emails_8w
Categorical (6): payment_type, employment_status, housing_status, source, device_os, email_is_free
Baseline ROC AUC (valid): 0.8913


In [7]:
LGBM_CAT_COLS = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
    "email_is_free",
]
assert set(LGBM_CAT_COLS) <= set(CAT_COLS)

X_train_lgbm = X_train.copy()
X_valid_lgbm = X_valid.copy()

# Level kategori = gabungan nilai teramati di train+valid (bukan dari label),
# lalu dtype yang sama dipakai di kedua split agar kode kategori konsisten.
for col in LGBM_CAT_COLS:
    observed = pd.concat([X_train[col], X_valid[col]], ignore_index=True)
    dtype = pd.CategoricalDtype(categories=pd.Index(observed.unique()).sort_values())
    X_train_lgbm[col] = X_train_lgbm[col].astype(dtype)
    X_valid_lgbm[col] = X_valid_lgbm[col].astype(dtype)

assert X_train_lgbm.columns.tolist() == X_valid_lgbm.columns.tolist() == FEATURES
for col in LGBM_CAT_COLS:
    assert X_train_lgbm[col].dtype == X_valid_lgbm[col].dtype
# Numerik boleh NaN (LightGBM menangani missing native); kategorikal tidak.
assert X_train_lgbm[LGBM_CAT_COLS].isna().sum().sum() == 0

n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
scale_pos_weight = n_neg / n_pos

lgbm = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
)
lgbm.fit(X_train_lgbm, y_train)
valid_proba_lgbm = lgbm.predict_proba(X_valid_lgbm)[:, 1]
auc_lgbm = roc_auc_score(y_valid, valid_proba_lgbm)

print(f"scale_pos_weight = {n_neg:,} / {n_pos:,} = {scale_pos_weight:.2f}")
print(f"LightGBM ROC AUC (valid): {auc_lgbm:.4f}")

[LightGBM] [Info] Number of positive: 3896, number of negative: 393143
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013905 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3206
[LightGBM] [Info] Number of data points in the train set: 397039, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.009813 -> initscore=-4.614223
[LightGBM] [Info] Start training from score -4.614223


scale_pos_weight = 393,143 / 3,896 = 100.91
LightGBM ROC AUC (valid): 0.8992


In [8]:
print("ROC AUC diagnostics (validation, month == 3)")
print(f"  Baseline  LogisticRegression : {auc_baseline:.4f}")
print(f"  Model     LightGBM           : {auc_lgbm:.4f}")
print(f"  Delta (LightGBM - baseline)  : {auc_lgbm - auc_baseline:+.4f}")

ROC AUC diagnostics (validation, month == 3)
  Baseline  LogisticRegression : 0.8913
  Model     LightGBM           : 0.8992
  Delta (LightGBM - baseline)  : +0.0079


In [9]:
def tpr_at_fpr_limit(y_true, y_score, limit=0.05):
    """TPR pada titik ROC terakhir dengan FPR <= limit."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score, dtype=float)

    if y_true.ndim != 1 or y_score.ndim != 1 or len(y_true) != len(y_score):
        raise ValueError("y_true dan y_score harus vektor 1D dengan panjang sama")
    if len(y_true) == 0 or not np.isin(y_true, [0, 1]).all():
        raise ValueError("y_true harus array biner non-empty")
    if np.unique(y_true).size != 2:
        raise ValueError("y_true harus memuat kedua kelas")
    if not np.isfinite(y_score).all():
        raise ValueError("y_score hanya boleh berisi nilai finite")
    if not 0 <= limit <= 1:
        raise ValueError("limit harus di rentang [0, 1]")

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    idx = np.flatnonzero(fpr <= limit)[-1]
    return float(tpr[idx]), float(fpr[idx]), float(thresholds[idx])

# Synthetic self-check: perfect ranking gives TPR=1 at FPR=0.
_test = tpr_at_fpr_limit([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9])
assert _test[0] == 1.0 and _test[1] == 0.0, _test
for _bad in (
    lambda: tpr_at_fpr_limit([0, 0], [0.2, 0.3]),
    lambda: tpr_at_fpr_limit([0, 1], [0.2, np.nan]),
    lambda: tpr_at_fpr_limit([0, 1], [0.2, 0.3], limit=1.1),
):
    try:
        _bad()
    except ValueError:
        pass
    else:
        raise AssertionError("invalid metric input was accepted")

print("TPR@FPR5% self-checks passed")

TPR@FPR5% self-checks passed


In [10]:
tpr_lgbm, fpr_lgbm, threshold_lgbm = tpr_at_fpr_limit(y_valid, valid_proba_lgbm)

print("Operating point LightGBM (validation, month == 3)")
print(f"  TPR @ FPR <= 5% : {tpr_lgbm:.4f}")
print(f"  FPR             : {fpr_lgbm:.4%}")
print(f"  Threshold       : {threshold_lgbm:.6f}")

with MODEL_PATH.open("wb") as f:
    pickle.dump(lgbm, f)
with MODEL_PATH.open("rb") as f:
    lgbm_reloaded = pickle.load(f)

proba_reloaded = lgbm_reloaded.predict_proba(X_valid_lgbm.iloc[:100])[:, 1]
np.testing.assert_array_equal(proba_reloaded, valid_proba_lgbm[:100])

size_bytes = MODEL_PATH.stat().st_size
assert size_bytes > 0
print(f"\nModel tersimpan: {MODEL_PATH.resolve()} ({size_bytes:,} bytes)")
print("Reload cocok: 100/100 probabilitas validasi identik")

Operating point LightGBM (validation, month == 3)
  TPR @ FPR <= 5% : 0.5223
  FPR             : 4.9818%
  Threshold       : 0.801168

Model tersimpan: D:\Cakrawala\SMT 5\MLOPs\P-1 Build Prediction Model\model.pkl (1,750,151 bytes)
Reload cocok: 100/100 probabilitas validasi identik
